# Evaluate the night-before forecast

This notebook reruns the Phase 2 evaluation: it trains on the first 300 days of the fictional hotel's history, tests on the last 65 (21 September to 24 November 2026), and compares the forecast with the same-weekday average of the previous four weeks. It runs on a CPU in a few seconds, locally or in Colab. Every result is simulated on fictional data.

## Set up

In Colab, the first cell clones the repository and installs the packages. Locally, open this notebook from the `notebooks` folder with the project's `.venv` selected, and the cell only adds `src` to the path.

In [ ]:
import os, sys

if 'google.colab' in sys.modules and not os.path.exists('BuffetIQ'):
    !git clone -b claude/zen-keller-p86mcy https://github.com/stamedytweet-bit/BuffetIQ.git
    %cd BuffetIQ
    !pip install -q -r requirements.txt
root = os.getcwd() if os.path.exists('config.yaml') else os.path.abspath('..')
sys.path.insert(0, os.path.join(root, 'src'))
os.chdir(root)

## Train, test and compare with the baseline

In [ ]:
import pandas as pd
from buffetiq import forecast

r = forecast.evaluate()
d = r['diners']
pd.DataFrame({'Random Forest': d['model'], 'Same weekday, last 4 weeks': d['baseline']}).round(2)

## Dishes, scored against the simulation's true demand

Croissants are in pieces; every other dish is in kilograms.

In [ ]:
rows = []
for v in r['dishes'].values():
    rows.append({'Dish': v['name'], 'MAE': v['model']['mae'], 'RMSE': v['model']['rmse'], 'MAPE %': v['model']['mape'],
                 'Baseline MAE': v['baseline']['mae'], 'Baseline MAPE %': v['baseline']['mape'],
                 'Rate method': v['rate_choice'], '80th percentile coverage': v['p80_coverage']})
pd.DataFrame(rows).round(2)

## Arrival curve: model or simple profile?

In [ ]:
r['curve']

## Charts

In [ ]:
from IPython.display import Image, display
for name in ('phase2_diners_test', 'phase2_feature_importance'):
    display(Image(filename=f'docs/charts/{name}.png'))

## Tomorrow's shopping list for the demo morning (Tuesday 24 November 2026)

In [ ]:
pd.Series(r['demo_morning']['shopping_list'], name='Amount')